# Lung: complete cell–ECM experiment

Recompute the six-ROI analysis, replay the fold-specific selected settings,
and fit both mouse-held-out reconstruction folds. All three contexts and the
label-prior baseline use five corruption realisations. The final section adds
ROI-held-out, block/isolated-corruption and training-fold ECM-partition controls.

**Estimated runtime:** 2–5 minutes for the primary analysis; allow 5–10 minutes
including the supplementary controls on an eight-thread CPU. Installation
and the first data download are excluded. The complete notebook uses CPU only.

The selected graph, feature, classifier and cleaning settings were chosen
within each original training fold. Their inner searches are not rerun.

## Setup

On Colab, keep **Latest** and select **CPU**. Run this cell, restart the
session if requested, then run all cells. Locally, install
`lung_requirements.txt` and keep the companion files beside the notebook.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

if not Path("experiment_setup.py").exists():
    urlretrieve(
        "https://raw.githubusercontent.com/moeghaf/mantpy_reproducibility/"
        "main/notebooks/experiments/experiment_setup.py",
        "experiment_setup.py",
    )
from experiment_setup import prepare_experiment
prepare_experiment("lung")

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mantpy as mt
from threadpoolctl import threadpool_limits
from lung_analysis.graphs import GraphRecipe
from lung_analysis.pipeline import GraphCache
from lung_analysis.realisations import make_realisation
import lung_helpers as H

SEED = 0
PATCH_SIZE = 10
WALL_CLUSTER = 1
EVALUATION_SEEDS = list(range(5))
KINDS = ("mixed", "block", "isolated")
RUN_SUPPLEMENTARY = True
RUN_TARGET_REFIT_CONTROL = True
RESULTS = Path("results/lung")
RESULTS.mkdir(parents=True, exist_ok=True)
threadpool_limits(limits=int(os.environ.get("OMP_NUM_THREADS", "8")))
np.random.seed(SEED)
selected = H.load_selections()
stage_times = {}
experiment_started = time.perf_counter()

## Load all six ROIs

In [ ]:
started = time.perf_counter()
cache_dir = Path(os.environ.get("MANTPY_CACHE", "data"))
bundle = mt.datasets.balbc_pbs_lung(cache_dir=cache_dir)
cells = bundle.cohort_cells
SHOWCASE = bundle.showcase_roi
assert len(cells) == 6 and bundle.metadata["Mouse"].nunique() == 2
stage_times["data_loading"] = time.perf_counter() - started
bundle.metadata[["sample_id", "Mouse", "strain", "treatment"]]

In [ ]:
panel = bundle.panel.loc[bundle.panel["keep"].eq(1)].reset_index(drop=True)
fn_index = int(panel.index[panel["name"].eq("Fibronectin")][0])
raw_fn = np.asarray(bundle.images[SHOWCASE][fn_index], dtype=float)
vmax = np.percentile(raw_fn[raw_fn > 0], 99)
aec = cells[SHOWCASE].obs["cell_type"].astype(str).eq("AEC").to_numpy()
xy = np.asarray(cells[SHOWCASE].obsm["spatial"])
plt.figure(figsize=(6.5, 6.5))
plt.imshow(raw_fn, cmap="gray", vmin=0, vmax=vmax)
plt.scatter(xy[aec, 0], xy[aec, 1], s=5, c="#D55E00", label="AEC")
plt.xlabel("x (µm)")
plt.ylabel("y (µm)")
plt.title(f"{SHOWCASE} · fibronectin and airway epithelial cells")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Recompute the selected ECM partition

Standardise each marker over the pooled pixels, extract ten-pixel patches,
retain patches strictly above each ROI's minimum mean feature value,
fit four K-means components and remove the lowest-signal component.
Signal clusters are ordered by size. This is a fresh fit.

In [ ]:
started = time.perf_counter()
ecm = mt.pp.ecm_patches_from_images(
    bundle.images, bundle.panel, normalize="pooled_zscore",
    patch_size=PATCH_SIZE, features=["mean"],
    ecm_K=None, background_quantile=0.0,
)
primary_support = H.retain_primary_patch_support(ecm)
partition = mt.pp.cluster_ecm_patches(
    ecm, n_clusters=4, method="kmeans", subset="all",
    random_state=SEED, n_init=10,
)
background = mt.pp.remove_background_patches(
    ecm, partition, background="lowest_mean", order_signal_by="size",
)
cohort = H.cohort_from_mantpy(bundle, ecm, PATCH_SIZE)
total = sum(len(r.label) for r in cohort.rois.values())
signal = sum(int((r.label >= 0).sum()) for r in cohort.rois.values())
stage_times["partition"] = time.perf_counter() - started
print(f"{total:,} patches; {signal:,} signal patches in three ECM clusters.")
print("Paper reference: 27,318 patches; 11,435 signal patches.")

In [ ]:
mt.pl.ecm_centroid_heatmap(
    ecm, ecm_names=bundle.ecm_names, K_ecm=3,
    cluster_colors=bundle.ecm_palette, title="ECM marker profiles",
    figsize=(8, 4),
)
H.plot_map(
    cohort.rois[SHOWCASE], cohort.rois[SHOWCASE].label,
    f"{SHOWCASE} · fresh ECM clusters", bundle.ecm_palette,
)

## Descriptive graphs and associations

The reference graph connects cells within 15 pixels, adjacent ECM patches
on an eight-connected grid, and each cell to its five nearest patches within
15 pixels. Background patches remain unlabelled nodes and are excluded from
association counts.

In [ ]:
started = time.perf_counter()
mt.pp.attach_ecm_patches(cells, ecm)
graph_result = mt.gr.build_cell_ecm_graphs(
    cells, cell_graph="squidpy", cell_radius=(0, 15),
    ecm_edge_method="grid", ecm_grid_connectivity=8,
    cell_ecm_edge_method="knn", cell_ecm_k=5, cell_ecm_Dmax=15,
)
graph_counts = pd.DataFrame([vars(row) for row in graph_result.summaries])
cache = GraphCache(96)
reference = GraphRecipe.from_dict(selected["reference_graph"])
association, per_roi_association = H.descriptive_association(cohort, cache, reference)
stage_times["descriptive_graphs_and_association"] = time.perf_counter() - started
graph_counts

In [ ]:
matrix = association.pivot(index="cell_type", columns="cluster", values="log2_enr")
plt.figure(figsize=(5, 6))
plt.imshow(matrix.to_numpy(), cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
plt.xticks(range(3), ["ECM 0", "ECM 1", "ECM 2"])
plt.yticks(range(len(matrix)), matrix.index)
plt.colorbar(label="log₂(observed / expected)")
plt.title("Cell-type contacts with ECM · all six ROIs")
plt.tight_layout()
plt.show()

In [ ]:
aec_effects = per_roi_association.query("cell_type == 'AEC'").copy()
mt.pl.cell_ecm_enrichment_per_roi(
    aec_effects, cluster_colors=bundle.ecm_palette,
    title="AEC contacts · each ROI", figsize=(7, 3.5),
)
aec_effects[["roi", "mouse", "cluster", "n_obs", "n_exp", "log2_enr"]]

## Matrix contacted by airway epithelial cells

Within ECM 1, count each patch once. Subtract the mean marker profile of
uncontacted patches from that of AEC-contacted patches within each ROI, then
average the six ROIs with equal weight.

In [ ]:
contacts = H.aec_contacts(cohort, cache, reference)
contrasts = H.marker_contrasts(cohort, contacts, WALL_CLUSTER)
mean_contrast = contrasts.mean(axis=0)
mean_contrast.plot.bar(
    figsize=(8, 3.5), color=np.where(mean_contrast >= 0, "#009E73", "#0072B2"),
)
plt.axhline(0, color="gray", linewidth=0.7)
plt.ylabel("AEC-contacted − uncontacted\nstandardised marker intensity")
plt.title("Bronchovascular-wall matrix · equal-weight mean of six ROIs")
plt.tight_layout()
plt.show()
contrasts.round(3)

In [ ]:
roi = cohort.rois[SHOWCASE]
hit = contacts[SHOWCASE] & (roi.label == WALL_CLUSTER)
overlay = np.zeros(raw_fn.shape, dtype=bool)
for x, y in roi.patch_xy[hit]:
    x0, y0 = int(x - PATCH_SIZE / 2), int(y - PATCH_SIZE / 2)
    overlay[y0:y0 + PATCH_SIZE, x0:x0 + PATCH_SIZE] = True
plt.figure(figsize=(6.5, 6.5))
plt.imshow(raw_fn, cmap="gray", vmin=0, vmax=vmax)
plt.imshow(np.ma.masked_where(~overlay, overlay), cmap="Greens",
           vmin=0, vmax=1, alpha=0.55)
plt.scatter(xy[aec, 0], xy[aec, 1], s=5, c="#D55E00", label="AEC")
plt.xlabel("x (µm)")
plt.ylabel("y (µm)")
plt.title(f"{SHOWCASE} · AEC-contacted bronchovascular matrix")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Sensitivity to the cell–ECM edge rule

Recompute the 48 seeded graph recipes and reference recipe used for the
descriptive sensitivity analysis. These are effect-size comparisons.

In [ ]:
started = time.perf_counter()
sensitivity = H.graph_sensitivity(cohort, selected, RESULTS)
stage_times["graph_sensitivity"] = time.perf_counter() - started
for cluster in range(3):
    subset = sensitivity.loc[sensitivity.cluster.eq(cluster)]
    plt.scatter(subset.recipe_index, subset.log2_enr,
                s=18, c=bundle.ecm_palette[cluster], label=f"ECM {cluster}")
plt.axhline(0, color="gray", linewidth=0.7)
plt.xlabel("Graph recipe · 0 is the reference")
plt.ylabel("AEC log₂(observed / expected)")
plt.title("AEC association across 49 graph recipes")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## Replay the mouse-held-out selections

Each mouse has its own selected graph, feature, classifier and cleaning
recipe. Fit on the other mouse's three pristine ROIs. Training-only
standardisation and any selected temperature calibration are refitted.
The patch's own ECM label is excluded from its context features.

In [ ]:
H.selection_table(selected, "mouse").round(4)

In [ ]:
protocol = {"benchmark": selected["benchmark"]}
realisations = {
    (kind, seed): make_realisation(cohort, "forward", kind, seed, protocol)
    for kind in KINDS for seed in EVALUATION_SEEDS
}
for (_, seed), real in realisations.items():
    for name, corruption in real.items():
        expected = round(0.1 * int((cohort.rois[name].label >= 0).sum()))
        assert int(corruption.is_artifact.sum()) == expected
print("Five independent evaluation realisations for each corruption type.")

In [ ]:
started = time.perf_counter()
primary_rows, primary_classes, supplementary_mouse_rows = [], [], []
fold_records, examples, primary_fits = [], {}, {}
for held_out in cohort.mice():
    for context in H.CONTEXTS:
        fitted = H.fit_fold(
            cohort, selected, "mouse", held_out, context,
            cache, RESULTS / "checkpoints",
        )
        scores, class_counts, maps = H.score_fold(
            fitted, cohort, realisations, cache, kinds=("mixed",),
        )
        primary_rows.append(scores)
        primary_classes.append(class_counts)
        fold_records.append(H.fold_record(fitted))
        primary_fits[(held_out, context)] = fitted
        if context == "cell":
            examples.update(maps)
        print(f"{held_out} · {context}: {len(scores)} held-out scores")
primary = pd.concat(primary_rows, ignore_index=True)
class_counts = pd.concat(primary_classes, ignore_index=True)
assert len(primary) == 120
assert primary.groupby(["context", "roi"]).size().eq(5).all()
per_roi_scores, score_summary = H.summarise_scores(primary)
per_roi_recovery, recovery_summary = H.summarise_class_recovery(class_counts)
stage_times["mouse_held_out_fitting_and_scoring"] = time.perf_counter() - started
H.plot_detection(per_roi_scores)
score_summary.xs("mean", axis=1, level=1).round(4)

In [ ]:
# Fresh outcomes beside rounded manuscript reference values.
fresh = per_roi_scores.groupby("context")[list(H.METRICS)].mean()
reference_values = pd.DataFrame.from_dict(selected["paper_reference"], orient="index")
comparison = pd.concat(
    {"fresh fit": fresh[reference_values.columns],
     "paper reference": reference_values}, axis=1,
)
comparison.round(4)

In [ ]:
recovery_summary.query("context == 'cell' and kind == 'mixed'")[
    ["cluster", "mean", "std", "min", "max"]
].round(4)

## One held-out reconstruction

In [ ]:
example_roi = next(iter(examples))
for key, label in (
    ("pristine", "Pristine ECM labels"),
    ("observed", "10% mixed label corruption"),
    ("cleaned", "Cell-context reconstruction"),
):
    H.plot_map(
        cohort.rois[example_roi], examples[example_roi][key],
        f"{example_roi} · {label}", bundle.ecm_palette,
    )

## Supplementary controls

Reuse the mouse-held-out models for block-only and isolated corruption.
Then refit the saved ROI-held-out configurations and evaluate all three
corruption types. The folds and their selected settings remain separate.
Set `RUN_SUPPLEMENTARY=False` above to run the primary experiment alone.

In [ ]:
started = time.perf_counter()
control_rows, control_classes = [], []
if RUN_SUPPLEMENTARY:
    for fitted in primary_fits.values():
        scores, cls, _ = H.score_fold(
            fitted, cohort, realisations, cache, kinds=("block", "isolated"),
        )
        control_rows.append(scores)
        control_classes.append(cls)
    for held_out in cohort.roi_names:
        for context in H.CONTEXTS:
            fitted = H.fit_fold(
                cohort, selected, "roi", held_out, context,
                cache, RESULTS / "checkpoints",
            )
            scores, cls, _ = H.score_fold(fitted, cohort, realisations, cache, kinds=KINDS)
            control_rows.append(scores)
            control_classes.append(cls)
            fold_records.append(H.fold_record(fitted))
        print(f"ROI-held-out controls: {held_out}")
controls = pd.concat(control_rows, ignore_index=True) if control_rows else pd.DataFrame()
control_class_counts = pd.concat(control_classes, ignore_index=True) if control_classes else pd.DataFrame()
stage_times["supplementary_controls"] = time.perf_counter() - started
if RUN_SUPPLEMENTARY:
    assert len(controls) == 600
    all_scores = pd.concat([primary, controls], ignore_index=True)
    all_per_roi, all_summary = H.summarise_scores(all_scores)
    H.plot_detection(all_per_roi, scheme="roi")
    display(all_summary.xs("mean", axis=1, level=1)[
        ["roc_auc", "prior_adjusted_roc_auc", "recovery", "accuracy_after"]
    ].round(4))

## Refit the ECM target partition within each training fold

Replay the saved signal K and classifier configuration for each of six
held-out ROIs, separately for training-cohort and per-ROI marker scaling.
Fit foreground and cluster centres on the five training ROIs, then predict
the held-out partition. The native grid contains all 4,554 tiles per ROI.

Both scaling arms receive identical physical corruption proposals on their
common foreground. Blocks can propose a label already present, so actual
corruption can be below 10%; its realised rate is reported. The per-ROI arm
uses its held-out image's pixel moments as an unsupervised transformation.
Neither arm uses held-out targets to fit centres or classifiers.

Set `RUN_TARGET_REFIT_CONTROL=False` to skip this additional control.

In [ ]:
from lung_analysis import target_refit as T

target_spec = selected["target_refit_control"]
pd.DataFrame([
    {"normalization": norm, "held_out": name,
     "selected_signal_K": saved["selected_signal_k"],
     "training_ROIs": len(saved["training_rois"])}
    for norm, folds in target_spec["target_definition"].items()
    for name, saved in folds.items()
])

In [ ]:
target_configuration_table = pd.concat([
    H.selection_table({"roi": folds}, "roi").assign(normalization=norm)
    for norm, folds in target_spec["normalization_selection"].items()
], ignore_index=True)
target_configuration_table.round(4)

In [ ]:
started = time.perf_counter()
target_rows, target_classes, target_records, target_fold_records = [], [], [], []
if RUN_SUPPLEMENTARY and RUN_TARGET_REFIT_CONTROL:
    raw_stats = T.raw_statistics(bundle, PATCH_SIZE)
    geometry = T.geometry_from_bundle(bundle, raw_stats, PATCH_SIZE)
    target_dir = RESULTS / "training_fold_targets"
    target_dir.mkdir(parents=True, exist_ok=True)
    for held_out in geometry.roi_names:
        target_cohorts, target_caches = {}, {}
        for normalization in T.NORMALIZATIONS:
            native, native_cache, target_record = T.refit_targets(
                geometry, raw_stats, selected, normalization, held_out,
                RESULTS / "checkpoints",
            )
            target_cohorts[normalization] = native
            target_caches[normalization] = native_cache
            target_records.append(target_record)
            H.export_partition(
                native, target_dir / f"{normalization}_{held_out}_partition.csv",
            )
        paired = T.paired_corruptions(
            target_cohorts, held_out, kinds=KINDS, seeds=EVALUATION_SEEDS,
        )
        for normalization, native in target_cohorts.items():
            native_cache = target_caches[normalization]
            saved_folds = target_spec["normalization_selection"][normalization]
            for context in ("cell", "ecm", "joint"):
                fitted = H.fit_fold(
                    native, {"roi": saved_folds}, "roi", held_out, context,
                    native_cache, RESULTS / "checkpoints",
                )
                measured, cls, example = T.score_target_fold(
                    fitted, native, paired, native_cache, normalization,
                )
                target_rows.append(measured)
                target_classes.append(cls)
                record = H.fold_record(fitted) | {"normalization": normalization}
                target_fold_records.append(record)
                np.savez_compressed(
                    target_dir / f"{normalization}_{held_out}_{context}_mixed_seed0.npz",
                    **example,
                )
        print(f"Training-fold targets: {held_out} · two scaling arms")
target_scores = pd.concat(target_rows, ignore_index=True) if target_rows else pd.DataFrame()
target_class_counts = pd.concat(target_classes, ignore_index=True) if target_classes else pd.DataFrame()
stage_times["training_fold_target_control"] = time.perf_counter() - started
if not target_scores.empty:
    assert len(target_scores) == 540 and len(target_fold_records) == 36
    assert len(target_records) == 12
    assert target_scores.groupby(
        ["normalization", "context", "outer_roi", "kind"]
    ).size().eq(5).all()
    target_per_roi, target_summary = T.summarise_target_scores(target_scores)
    display(target_summary[[
        "roc_auc", "prior_adjusted_roc_auc_full_signal", "recovery",
        "accuracy_after", "realized_corruption_fraction",
        "macro_f1_after_common_signal",
    ]].round(4))

In [ ]:
if not target_scores.empty:
    reference_target = pd.DataFrame(target_spec["paper_reference"]).set_index(
        ["normalization", "context", "kind"]
    )
    comparison_target = pd.concat({
        "fresh fit": target_summary[reference_target.columns],
        "paper reference": reference_target,
    }, axis=1)
    display(comparison_target.loc[
        pd.IndexSlice[:, "cell", "mixed"],
        pd.IndexSlice[:, ["roc_auc", "accuracy_after", "macro_f1_after_common_signal"]],
    ].round(4))
    mixed = target_per_roi.query("context == 'cell' and kind == 'mixed'")
    for i, normalization in enumerate(T.NORMALIZATIONS):
        values = mixed.loc[mixed.normalization.eq(normalization)]
        plt.scatter(
            i + np.linspace(-0.10, 0.10, len(values)),
            values["accuracy_after"], s=30,
            c="#0072B2" if i == 0 else "#009E73",
        )
        plt.hlines(values["accuracy_after"].mean(), i - 0.15, i + 0.15,
                   color="black", linewidth=1.5)
    plt.xticks([0, 1], ["Training-cohort scaling", "Per-ROI scaling"])
    plt.ylabel("Final label accuracy · each held-out ROI")
    plt.ylim(0.75, 1.01)
    plt.title("Cell-context reconstruction · training-fold ECM partitions")
    plt.tight_layout()
    plt.show()

## Save results and timings

In [ ]:
H.export_partition(cohort, RESULTS / "fresh_partition.csv")
association.to_csv(RESULTS / "celltype_association.csv", index=False)
per_roi_association.to_csv(RESULTS / "association_per_roi.csv", index=False)
contrasts.to_csv(RESULTS / "aec_marker_contrasts.csv")
graph_counts.to_csv(RESULTS / "descriptive_graph_counts.csv", index=False)
pd.DataFrame(primary_support).to_csv(RESULTS / "primary_patch_eligibility.csv", index=False)
sensitivity.to_csv(RESULTS / "aec_graph_sensitivity.csv", index=False)
primary.to_csv(RESULTS / "mouse_held_out_metrics.csv", index=False)
per_roi_scores.to_csv(RESULTS / "mouse_held_out_metrics_per_roi.csv", index=False)
class_counts.to_csv(RESULTS / "mouse_held_out_class_counts.csv", index=False)
recovery_summary.to_csv(RESULTS / "mouse_held_out_class_recovery.csv", index=False)
comparison.to_csv(RESULTS / "fresh_vs_paper_reference.csv")
if RUN_SUPPLEMENTARY:
    controls.to_csv(RESULTS / "supplementary_control_metrics.csv", index=False)
    control_class_counts.to_csv(RESULTS / "supplementary_class_counts.csv", index=False)
if not target_scores.empty:
    target_scores.to_csv(RESULTS / "training_fold_target_metrics.csv", index=False)
    target_per_roi.to_csv(RESULTS / "training_fold_target_metrics_per_roi.csv", index=False)
    target_class_counts.to_csv(RESULTS / "training_fold_target_class_counts.csv", index=False)
    comparison_target.to_csv(RESULTS / "training_fold_targets_fresh_vs_paper.csv")
    (RESULTS / "training_fold_target_provenance.json").write_text(json.dumps({
        "target_definitions": target_records,
        "classifier_folds": target_fold_records,
        "corruption_roi_seed": target_spec["corruption_seed"],
        "physical_proposals_paired_across_normalizations": True,
        "aggregation": "mean seeds within ROI; mean ROIs with equal weight",
        "optimisation_rerun": False,
    }, indent=2))
for name, maps in examples.items():
    np.savez_compressed(RESULTS / f"{name}_cell_context_example.npz", **maps)
stage_times["total_analysis_and_export"] = time.perf_counter() - experiment_started
provenance = {
    "scope": "Fresh full lung fits using published fold-specific selections",
    "optimisation_rerun": False,
    "identity": H.run_identity(cohort),
    "roi_order": cohort.roi_names,
    "evaluation_seeds": EVALUATION_SEEDS,
    "corruption_roi_seed": "evaluation_seed * 1000 + ROI index",
    "fit_seed": 0,
    "primary_fits": 8,
    "primary_score_rows": len(primary),
    "supplementary_fits": 24 if RUN_SUPPLEMENTARY else 0,
    "supplementary_score_rows": len(controls),
    "training_fold_target_kmeans_fits": len(target_records),
    "training_fold_target_classifier_fits": len(target_fold_records),
    "training_fold_target_score_rows": len(target_scores),
    "aggregation": "mean seeds within ROI; mean ROIs with equal weight",
    "class_recovery": "sum counts across seeds within ROI; equal-weight ROI mean",
    "partition": {"total_patches": total, "signal_patches": signal,
                  "source": "fresh Mantpy K=4 fit",
                  "patch_eligibility": primary_support},
    "folds": fold_records,
    "stage_times_seconds": stage_times,
    "controls_not_recomputed": selected["unsupported_controls"],
}
(RESULTS / "run_provenance.json").write_text(json.dumps(provenance, indent=2))
print(f"Saved results to {RESULTS.resolve()}")
pd.Series(stage_times, name="seconds").round(2)